# 07 Results Freeze

Final reviewer evidence matrix and artifact freeze.

## Setup

Clone/pull latest code into `/content/ECG-RAMBA`, restore verified Drive mirror artifacts, and define a logging command helper.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import time

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print(f'Drive mount skipped or already mounted: {exc}')

DRIVE_ROOT = Path('/content/drive/MyDrive/ECG-Ramba')
MIRROR_REVISION_ROOT = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision'
REPO_URL = 'https://github.com/BrianNguyen29/ECG-RAMBA.git'
BRANCH = os.environ.get('ECG_RAMBA_BRANCH', 'main')
REPO_DIR = Path('/content/ECG-RAMBA')

os.chdir('/content')
if REPO_DIR.exists() and not (REPO_DIR / '.git').exists():
    raise RuntimeError(f'{REPO_DIR} exists but is not a git checkout. Rename it or use a fresh runtime.')
if not REPO_DIR.exists():
    print(f'$ git clone --branch {BRANCH} {REPO_URL} "{REPO_DIR}"')
    subprocess.run(f'git clone --branch {BRANCH} {REPO_URL} "{REPO_DIR}"', shell=True, check=True)

os.chdir(REPO_DIR)

def run(cmd, check=True, log_path=None, tail_lines=120, cwd=None):
    run_cwd = Path(cwd) if cwd else Path.cwd()
    print(f'$ {cmd}', flush=True)
    if log_path is None:
        log_dir = run_cwd / 'reports' / 'revision' / 'logs'
        log_dir.mkdir(parents=True, exist_ok=True)
        log_path = log_dir / f'notebook_command_{time.strftime("%Y%m%d_%H%M%S")}.log'
    else:
        log_path = Path(log_path)
        if not log_path.is_absolute():
            log_path = run_cwd / log_path
        log_path.parent.mkdir(parents=True, exist_ok=True)

    with log_path.open('w', encoding='utf-8') as log:
        proc = subprocess.Popen(
            cmd,
            shell=True,
            cwd=str(run_cwd),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding='utf-8',
            errors='replace',
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        rc = proc.wait()
    print(f'Command log: {log_path}')
    if check and rc:
        lines = log_path.read_text(encoding='utf-8', errors='replace').splitlines()
        for line in lines[-tail_lines:]:
            print(line)
        raise subprocess.CalledProcessError(rc, cmd)
    return subprocess.CompletedProcess(cmd, rc)

run('git fetch origin', check=False)
run(f'git checkout {BRANCH}', check=True)
run(f'git pull --ff-only --autostash origin {BRANCH}', check=True)

if MIRROR_REVISION_ROOT.exists():
    run(
        f'python -u scripts/revision/artifact_mirror.py restore --mirror-root "{MIRROR_REVISION_ROOT}" --replace-mismatched',
        log_path='reports/revision/logs/final_evidence_restore.log',
    )
else:
    print('Mirror not found; continuing with repo-local reports/revision:', MIRROR_REVISION_ROOT)

print('cwd       :', Path.cwd())
print('drive_root:', DRIVE_ROOT)
print('repo_dir  :', REPO_DIR)
print('branch    :', BRANCH)
run('git rev-parse --short HEAD', check=False)
run('git status --short --branch', check=False)


## Required Input Check

Validate the revision-plan CSV files and the final evidence inputs before building the final matrix. This fails early if paired ResNet or other required artifacts are missing from the restored mirror.


In [ ]:
import pandas as pd

revision_root = Path('reports/revision')
plan_csvs = [
    Path('docs/revision_plan/experiment_registry.csv'),
    Path('docs/revision_plan/claim_evidence_map.csv'),
    Path('docs/revision_plan/task_board.csv'),
]
for csv_path in plan_csvs:
    df = pd.read_csv(csv_path)
    print(f'{csv_path}: rows={len(df)} cols={len(df.columns)}')

required_final_inputs = {
    'calibration': revision_root / 'metrics' / 'calibration_ci_oof_final_ema_predictions.json',
    'pooling': revision_root / 'metrics' / 'pooling_sensitivity.csv',
    'baseline': revision_root / 'metrics' / 'baseline_summary.csv',
    'component': revision_root / 'metrics' / 'component_check_summary.json',
    'hrv_domain': revision_root / 'metrics' / 'hrv_domain_summary.csv',
    'robustness': revision_root / 'metrics' / 'robustness_summary.csv',
    'paired_minirocket': revision_root / 'metrics' / 'paired_full_vs_minirocket_comparison.json',
    'paired_resnet': revision_root / 'metrics' / 'paired_full_vs_resnet_comparison.json',
    'a0_status': revision_root / 'a0_resolution_status.json',
    'claim_map': Path('docs/revision_plan/claim_evidence_map.csv'),
    'task_board': Path('docs/revision_plan/task_board.csv'),
}
optional_final_inputs = {
    'paired_raw_mamba': revision_root / 'metrics' / 'paired_full_vs_raw_mamba_comparison.json',
    'paired_transformer': revision_root / 'metrics' / 'paired_full_vs_transformer_comparison.json',
    'paired_hybrid_morphology': revision_root / 'metrics' / 'paired_full_vs_hybrid_morphology_comparison.json',
    'hybrid_morphology_summary': revision_root / 'metrics' / 'hybrid_morphology_baseline_summary.json',
    'claim_readiness_gates': revision_root / 'metrics' / 'claim_readiness_gates.json',
    'claim_readiness_table': revision_root / 'tables' / 'table_claim_readiness_gates.csv',
    'external_protocol_gate_summary': revision_root / 'metrics' / 'external_protocol_gate_summary.csv',
    'representation_evidence_status': revision_root / 'metrics' / 'representation_evidence_status.json',
    'representation_probe_summary': revision_root / 'metrics' / 'representation_probe_summary.json',
    'representation_probe_table': revision_root / 'tables' / 'table_representation_probe.csv',
    'representation_cka_table': revision_root / 'tables' / 'table_representation_cka.csv',
    'representation_probe_manifest': revision_root / 'manifests' / 'representation_probe_manifest.json',
    'fewshot_ptbxl_summary': revision_root / 'metrics' / 'fewshot_ptbxl_summary.csv',
    'fewshot_ptbxl_table': revision_root / 'tables' / 'table_fewshot_ptbxl.csv',
    'fewshot_ptbxl_bootstrap': revision_root / 'metrics' / 'fewshot_ptbxl_bootstrap.json',
    'fewshot_ptbxl_manifest': revision_root / 'manifests' / 'fewshot_ptbxl_run_manifest.json',
    'fewshot_cpsc2021_summary': revision_root / 'metrics' / 'fewshot_cpsc2021_summary.csv',
    'fewshot_cpsc2021_table': revision_root / 'tables' / 'table_fewshot_cpsc2021.csv',
    'fewshot_cpsc2021_bootstrap': revision_root / 'metrics' / 'fewshot_cpsc2021_bootstrap.json',
    'fewshot_cpsc2021_manifest': revision_root / 'manifests' / 'fewshot_cpsc2021_run_manifest.json',
    'fewshot_georgia_summary': revision_root / 'metrics' / 'fewshot_georgia_summary.csv',
    'fewshot_georgia_table': revision_root / 'tables' / 'table_fewshot_georgia.csv',
    'fewshot_georgia_bootstrap': revision_root / 'metrics' / 'fewshot_georgia_bootstrap.json',
    'fewshot_georgia_manifest': revision_root / 'manifests' / 'fewshot_georgia_run_manifest.json',
    'robustness_multicomparator_summary': revision_root / 'metrics' / 'robustness_multicomparator_summary.csv',
    'robustness_multicomparator_pairwise': revision_root / 'metrics' / 'robustness_multicomparator_pairwise.json',
    'robustness_multicomparator_table': revision_root / 'tables' / 'table_robustness_multicomparator.csv',
    'robustness_multicomparator_manifest': revision_root / 'manifests' / 'robustness_multicomparator_manifest.json',
}
def collect_missing_final_inputs(verbose=True):
    missing = []
    for label, path in required_final_inputs.items():
        exists = path.exists()
        if verbose:
            print(f'{label:18}: exists={exists} size={path.stat().st_size if exists else None} path={path}')
        if not exists:
            missing.append(f'{label}={path}')
    return missing

missing_inputs = collect_missing_final_inputs(verbose=True)
if missing_inputs:
    print('\nRequired inputs are missing from the active repo. Attempting verified Drive mirror restore before failing.')
    if 'MIRROR_REVISION_ROOT' not in globals():
        raise RuntimeError('MIRROR_REVISION_ROOT is undefined. Run the Setup cell first.')
    if not MIRROR_REVISION_ROOT.exists():
        raise FileNotFoundError(f'Mirror root does not exist: {MIRROR_REVISION_ROOT}')
    run(
        f'python -u scripts/revision/artifact_mirror.py restore --mirror-root "{MIRROR_REVISION_ROOT}" --replace-mismatched',
        log_path='reports/revision/logs/final_evidence_required_restore.log',
    )
    print('\nRechecking required final evidence inputs after mirror restore:')
    missing_inputs = collect_missing_final_inputs(verbose=True)
    if missing_inputs:
        print('\nVerified mirror manifest did not restore every required input. Trying direct path fallback from Drive mirror/repo roots.')
        import shutil

        def _candidate_source_roots():
            roots = []
            if 'MIRROR_REVISION_ROOT' in globals():
                roots.append(('verified_mirror_path', Path(MIRROR_REVISION_ROOT)))
            drive_root = globals().get('DRIVE_ROOT', Path('/content/drive/MyDrive/ECG-Ramba'))
            roots.append(('drive_repo_reports', Path(drive_root) / 'ECG-RAMBA' / 'reports' / 'revision'))
            roots.append(('drive_revision_artifacts', Path(drive_root) / 'revision_artifacts' / 'reports' / 'revision'))
            return roots

        restored_direct, unresolved_direct = [], []
        for label, destination in required_final_inputs.items():
            destination = Path(destination)
            if destination.exists() and destination.stat().st_size > 0:
                continue
            try:
                rel_to_revision = destination.relative_to(revision_root)
            except ValueError:
                unresolved_direct.append(f'{label}={destination} (not under reports/revision; direct fallback skipped)')
                continue
            copied = False
            for source_label, source_root in _candidate_source_roots():
                source = source_root / rel_to_revision
                if source.exists() and source.stat().st_size > 0:
                    destination.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(source, destination)
                    restored_direct.append(f'{label}: {source_label}:{source} -> {destination}')
                    copied = True
                    break
            if not copied:
                unresolved_direct.append(f'{label}={destination}')
        print(f'Direct final-input fallback restore: restored={len(restored_direct)} unresolved={len(unresolved_direct)}')
        for item in restored_direct:
            print(' - restored', item)
        for item in unresolved_direct:
            print(' - unresolved', item)
        print('\nRechecking required final evidence inputs after direct fallback restore:')
        missing_inputs = collect_missing_final_inputs(verbose=True)
if missing_inputs:
    raise FileNotFoundError('Missing required final evidence inputs after mirror restore/direct fallback: ' + '; '.join(missing_inputs))

# Cross-notebook provenance guard: Notebook 07 must not freeze final tables
# if calibration or paired comparisons still point to an older Full OOF file.
from scripts.revision.common import sha256_file

current_oof_path = revision_root / 'predictions' / 'oof_final_ema_predictions.npz'
current_freeze_path = revision_root / 'manifests' / 'oof_final_ema_freeze_manifest.json'
current_oof_sha = sha256_file(current_oof_path)
current_freeze_sha = sha256_file(current_freeze_path)
current_freeze = json.loads(current_freeze_path.read_text(encoding='utf-8'))
print('Current frozen OOF SHA     :', current_oof_sha)
print('Current freeze manifest SHA:', current_freeze_sha)

calibration_payload = json.loads(required_final_inputs['calibration'].read_text(encoding='utf-8'))
calibration_errors = []
if calibration_payload.get('predictions_sha256') != current_oof_sha:
    calibration_errors.append(
        'calibration predictions_sha256 mismatch: '
        f"json={calibration_payload.get('predictions_sha256')} current={current_oof_sha}"
    )
if calibration_payload.get('freeze_manifest_sha256') != current_freeze_sha:
    calibration_errors.append(
        'calibration freeze_manifest_sha256 mismatch: '
        f"json={calibration_payload.get('freeze_manifest_sha256')} current={current_freeze_sha}"
    )
expected_shape = {
    'y_prob': [current_freeze.get('validated_records'), current_freeze.get('n_classes')],
    'y_true': [current_freeze.get('validated_records'), current_freeze.get('n_classes')],
}
if calibration_payload.get('shape') != expected_shape:
    calibration_errors.append(f"calibration shape mismatch: json={calibration_payload.get('shape')} expected={expected_shape}")
if calibration_errors:
    print('\n'.join(' - ' + item for item in calibration_errors))
    raise RuntimeError(
        'Calibration artifact is stale for the current frozen OOF. '
        'Rerun Notebook 03, publish the mirror, then rerun Notebook 04 and Notebook 07.'
    )
print('Calibration provenance guard: OK')

paired_required = {
    'MiniRocket-only': required_final_inputs['paired_minirocket'],
    'ResNet1D/CNN': required_final_inputs['paired_resnet'],
}
paired_optional = {
    'Raw Mamba': optional_final_inputs['paired_raw_mamba'],
    'Transformer ECG': optional_final_inputs['paired_transformer'],
}

def _paired_protocol_compatible(label, payload):
    inputs = payload.get('inputs', {})
    freeze_input = inputs.get('freeze_manifest') or {}
    full_input = inputs.get('full_predictions') or {}
    full_metadata = full_input.get('metadata') or {}
    expected_records = int(current_freeze.get('validated_records', -1))
    expected_classes = int(current_freeze.get('n_classes', -1))
    expected_fingerprint = current_freeze.get('dataset_record_order_fingerprint')
    freeze_contract_ok = (
        freeze_input.get('checkpoint_kind') == 'final_ema'
        and int(freeze_input.get('validated_records', -1)) == expected_records
        and int(freeze_input.get('n_classes', -1)) == expected_classes
        and freeze_input.get('dataset_record_order_fingerprint') == expected_fingerprint
    )
    full_contract_ok = (
        full_metadata.get('checkpoint_kind') == 'final_ema'
        and full_metadata.get('dataset_record_order_fingerprint') == expected_fingerprint
        and full_metadata.get('protocol') == 'fold_final_ema_power_mean_v2_q3_threshold_0.5'
        and float(full_metadata.get('threshold', 0.5)) == 0.5
    )
    current_full_metrics = {}
    current_full_metrics.update(calibration_payload.get('metrics') or {})
    current_full_metrics.update(calibration_payload.get('calibration') or {})
    metric_mismatches = []
    for metric_name, row in (payload.get('metrics') or {}).items():
        if metric_name not in current_full_metrics:
            continue
        observed = row.get('full_value')
        expected = current_full_metrics.get(metric_name)
        if observed is None or expected is None:
            metric_mismatches.append(f'{metric_name}: missing observed={observed} expected={expected}')
            continue
        if abs(float(observed) - float(expected)) > 1e-12:
            metric_mismatches.append(f'{metric_name}: paired={observed} current={expected}')
    return freeze_contract_ok and full_contract_ok and not metric_mismatches, {
        'freeze_contract_ok': freeze_contract_ok,
        'full_contract_ok': full_contract_ok,
        'metric_mismatches': metric_mismatches,
    }


def _check_paired_payload(label, path, required=True):
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        if required:
            raise FileNotFoundError(f'Missing required paired comparison for {label}: {path}')
        print(f'Paired {label}: absent/deferred')
        return
    payload = json.loads(path.read_text(encoding='utf-8'))
    inputs = payload.get('inputs', {})
    full_sha = (inputs.get('full_predictions') or {}).get('sha256')
    freeze_sha = (inputs.get('freeze_manifest') or {}).get('sha256')
    if full_sha == current_oof_sha and freeze_sha == current_freeze_sha:
        print(f'Paired {label} provenance guard: OK')
        return
    compatible, details = _paired_protocol_compatible(label, payload)
    if compatible:
        print(
            f'Paired {label} provenance guard: stable-protocol reuse OK. '
            f'Container SHA changed full_sha={full_sha} current={current_oof_sha}; '
            f'freeze_sha={freeze_sha} current={current_freeze_sha}, but final_ema record/class/fingerprint contract '
            'and full metric values match current calibration.'
        )
        return
    raise RuntimeError(
        f'Paired {label} comparison is stale for current OOF. '
        f'full_sha={full_sha} current={current_oof_sha}; '
        f'freeze_sha={freeze_sha} current={current_freeze_sha}; details={details}. '
        'Rerun Notebook 04 paired comparison cells, then rerun Notebook 07.'
    )

for label, path in paired_required.items():
    _check_paired_payload(label, path, required=True)
for label, path in paired_optional.items():
    _check_paired_payload(label, path, required=False)

def restore_optional_final_inputs_from_drive():
    import shutil

    def _candidate_source_roots():
        roots = []
        if 'MIRROR_REVISION_ROOT' in globals():
            roots.append(('verified_mirror_path', Path(MIRROR_REVISION_ROOT)))
        drive_root = globals().get('DRIVE_ROOT', Path('/content/drive/MyDrive/ECG-Ramba'))
        roots.append(('drive_repo_reports', Path(drive_root) / 'ECG-RAMBA' / 'reports' / 'revision'))
        roots.append(('drive_revision_artifacts', Path(drive_root) / 'revision_artifacts' / 'reports' / 'revision'))
        return roots

    restored, still_missing = [], []
    for label, destination in optional_final_inputs.items():
        destination = Path(destination)
        if destination.exists() and destination.stat().st_size > 0:
            continue
        try:
            rel_to_revision = destination.relative_to(revision_root)
        except ValueError:
            still_missing.append(f'{label}={destination} (not under reports/revision)')
            continue
        copied = False
        for source_label, source_root in _candidate_source_roots():
            source = source_root / rel_to_revision
            if source.exists() and source.stat().st_size > 0:
                destination.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(source, destination)
                restored.append(f'{label}: {source_label}:{source} -> {destination}')
                copied = True
                break
        if not copied:
            drive_root = globals().get('DRIVE_ROOT', Path('/content/drive/MyDrive/ECG-Ramba'))
            flat_source = Path(drive_root) / 'final_evidence_tables' / destination.name
            if flat_source.exists() and flat_source.stat().st_size > 0:
                destination.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(flat_source, destination)
                restored.append(f'{label}: final_evidence_tables:{flat_source} -> {destination}')
                copied = True
        if not copied:
            still_missing.append(f'{label}={destination}')
    print(f'Optional final-input path fallback restore: restored={len(restored)} still_missing={len(still_missing)}')
    for item in restored:
        print(' - restored', item)
    if still_missing:
        print('Optional inputs still absent after fallback; these remain deferred unless their notebook cells are run:')
        for item in still_missing:
            print(' - missing', item)


restore_optional_final_inputs_from_drive()

print('Optional final evidence inputs:')
for label, path in optional_final_inputs.items():
    exists = path.exists()
    print(f'{label:18}: exists={exists} size={path.stat().st_size if exists else None} path={path}')

# Guard against stale Colab code: if few-shot artifacts exist, the final generator
# must know how to ingest them into C06 and claim_guidance.
generator_path = Path('scripts/revision/13_final_evidence_matrix.py')
generator_source = generator_path.read_text(encoding='utf-8')
required_generator_tokens = [
    'def summarize_fewshot',
    'def combine_fewshot_summaries',
    'fewshot_dataset_summaries',
    'fewshot_summary = combine_fewshot_summaries',
    '"fewshot_summary": fewshot_summary',
    '"fewshot": (',
    '"paired_transformer":',
    'transformer_paired_key_numbers',
]
missing_generator_tokens = [token for token in required_generator_tokens if token not in generator_source]
if missing_generator_tokens:
    raise RuntimeError(
        'Final evidence generator is stale and cannot ingest PTB-XL few-shot artifacts. '
        'Pull latest main / rerun Notebook 00 setup, then rerun Notebook 07. Missing tokens: '
        + ', '.join(missing_generator_tokens)
    )
print('Final evidence generator few-shot support: OK')

fewshot_datasets_for_final = ['ptbxl', 'cpsc2021', 'georgia']
fewshot_complete_datasets = []
fewshot_artifacts_present = False
for dataset in fewshot_datasets_for_final:
    dataset_paths = [
        optional_final_inputs[f'fewshot_{dataset}_summary'],
        optional_final_inputs[f'fewshot_{dataset}_table'],
        optional_final_inputs[f'fewshot_{dataset}_bootstrap'],
        optional_final_inputs[f'fewshot_{dataset}_manifest'],
    ]
    dataset_artifacts_present = all(path.exists() and path.stat().st_size > 0 for path in dataset_paths)
    print(f'Few-shot artifact set complete for {dataset}:', dataset_artifacts_present)
    if dataset_artifacts_present:
        fewshot_manifest = json.loads(optional_final_inputs[f'fewshot_{dataset}_manifest'].read_text(encoding='utf-8'))
        print(f'Few-shot manifest status for {dataset}:', fewshot_manifest.get('status'))
        if fewshot_manifest.get('status') != 'complete':
            raise RuntimeError(f'Few-shot manifest exists for {dataset} but is not complete; rerun Notebook 02 few-shot cell or remove stale artifact.')
        fewshot_complete_datasets.append(dataset)
fewshot_artifacts_present = bool(fewshot_complete_datasets)
print('Few-shot complete datasets:', fewshot_complete_datasets)

paired_resnet = json.loads(required_final_inputs['paired_resnet'].read_text(encoding='utf-8'))
resnet_metrics = paired_resnet.get('metrics', {})
print('Paired ResNet interpretations:')
for metric in ['pr_auc_macro', 'roc_auc_macro', 'f1_macro', 'brier_macro', 'ece_macro']:
    row = resnet_metrics.get(metric, {})
    print(f"  {metric}: {row.get('interpretation')} full={row.get('full_value')} comparator={row.get('comparator_value')}")

paired_raw_path = optional_final_inputs['paired_raw_mamba']
if paired_raw_path.exists():
    paired_raw = json.loads(paired_raw_path.read_text(encoding='utf-8'))
    raw_metrics = paired_raw.get('metrics', {})
    print('Paired Raw Mamba interpretations:')
    for metric in ['pr_auc_macro', 'roc_auc_macro', 'f1_macro', 'brier_macro', 'ece_macro']:
        row = raw_metrics.get(metric, {})
        print(f"  {metric}: {row.get('interpretation')} full={row.get('full_value')} comparator={row.get('comparator_value')}")
else:
    print('Paired Raw Mamba optional input is absent; final matrix will keep Raw Mamba incomplete until Notebook 04 produces it.')

expected_external_datasets = ['ptbxl', 'georgia', 'cpsc2021']
external_gate_path = optional_final_inputs['external_protocol_gate_summary']
if external_gate_path.exists():
    external_gate = pd.read_csv(external_gate_path)
    external_gate_columns = [
        'dataset',
        'status',
        'protocol_gate_passed',
        'manuscript_ready',
        'issue_count',
        'reused_existing',
        'gate_cache_key',
        'prediction_sha256',
        'slice_prediction_sha256',
        'gate_json',
        'gate_manifest',
    ]
    display_columns = [col for col in external_gate_columns if col in external_gate.columns]
    print('External protocol gate summary:')
    display(external_gate[display_columns])
    gate_pass_mask = external_gate.get('protocol_gate_passed', pd.Series(False, index=external_gate.index)).astype(str).str.lower().isin({'true', '1', 'yes'})
    present_datasets = sorted(external_gate['dataset'].astype(str).str.lower().tolist()) if 'dataset' in external_gate.columns else []
    passed_datasets = sorted(external_gate.loc[gate_pass_mask, 'dataset'].astype(str).str.lower().tolist()) if 'dataset' in external_gate.columns else []
    blocked_datasets = sorted(external_gate.loc[~gate_pass_mask, 'dataset'].astype(str).str.lower().tolist()) if 'dataset' in external_gate.columns else []
    deferred_datasets = [dataset for dataset in expected_external_datasets if dataset not in present_datasets]
    print('External gate passed datasets:', ', '.join(passed_datasets) if passed_datasets else 'none')
    print('External gate blocked datasets in summary:', ', '.join(blocked_datasets) if blocked_datasets else 'none')
    print('External deferred datasets absent from gate summary:', ', '.join(deferred_datasets) if deferred_datasets else 'none')
    print('External gate supporting artifacts:')
    for dataset in sorted(external_gate['dataset'].astype(str).tolist()) if 'dataset' in external_gate.columns else []:
        supporting = {
            'gate_json': revision_root / 'metrics' / f'external_{dataset}_protocol_gate.json',
            'label_mapping': revision_root / 'tables' / f'table_external_{dataset}_label_mapping.csv',
            'metrics_table': revision_root / 'tables' / f'table_external_{dataset}_metrics.csv',
            'gate_manifest': revision_root / 'manifests' / f'external_{dataset}_protocol_gate_manifest.json',
        }
        for label, path in supporting.items():
            print(f'  {dataset:8} {label:14}: exists={path.exists()} size={path.stat().st_size if path.exists() else None} path={path}')
else:
    print('External protocol gate summary absent; external datasets remain experimental/deferred in final evidence.')


## Claim Readiness Gates

Generate a lightweight blocker ledger for optional or unsupported claims before building the final matrix. This keeps full-HRV, clinical-readiness, broad-superiority, transformer, hybrid, and learned-comparator robustness wording auditable.


In [ ]:
from pathlib import Path
import shutil
import urllib.request

claim_readiness_script = Path('scripts/revision/28_claim_readiness_gates.py')
claim_readiness_required_tokens = [
    'Claim readiness gates for reviewer-sensitive ECG-RAMBA claims',
    'clinical_deployment_readiness',
    'robustness_learned_comparators',
    'full_hrv_feature_set',
]
CLAIM_READINESS_FALLBACK_SOURCE = '"""Claim readiness gates for reviewer-sensitive ECG-RAMBA claims.\n\nThis lightweight audit writes explicit blocker/status artifacts for claims that\ncannot be inferred from the main frozen OOF evaluation. It is intentionally\nconservative: a row is marked complete only when the required downstream\nartifacts already exist. Otherwise it records the missing evidence and the\nmanuscript-safe wording.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport json\nimport sys\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nPROJECT_ROOT = Path(__file__).resolve().parents[2]\nif str(PROJECT_ROOT) not in sys.path:\n    sys.path.insert(0, str(PROJECT_ROOT))\n\nfrom scripts.revision.common import (  # noqa: E402\n    MANIFEST_DIR,\n    METRIC_DIR,\n    TABLE_DIR,\n    ensure_revision_dirs,\n    git_commit,\n    save_json,\n    sha256_file,\n)\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\n        "--out-json",\n        type=Path,\n        default=METRIC_DIR / "claim_readiness_gates.json",\n    )\n    parser.add_argument(\n        "--out-table",\n        type=Path,\n        default=TABLE_DIR / "table_claim_readiness_gates.csv",\n    )\n    parser.add_argument(\n        "--out-manifest",\n        type=Path,\n        default=MANIFEST_DIR / "claim_readiness_gates_manifest.json",\n    )\n    return parser.parse_args()\n\n\ndef now_utc() -> str:\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef resolve(path: Path) -> Path:\n    return path if path.is_absolute() else PROJECT_ROOT / path\n\n\ndef rel(path: Path) -> str:\n    return resolve(path).resolve().relative_to(PROJECT_ROOT.resolve()).as_posix()\n\n\ndef present(path: Path) -> bool:\n    candidate = resolve(path)\n    return candidate.exists() and candidate.stat().st_size > 0\n\n\ndef status_from_required(required: list[Path], complete_status: str, blocked_status: str) -> tuple[str, list[str]]:\n    missing = [p.as_posix() for p in required if not present(p)]\n    return (complete_status, missing) if not missing else (blocked_status, missing)\n\n\ndef row(\n    *,\n    claim_id: str,\n    claim_area: str,\n    status: str,\n    manuscript_ready: bool,\n    evidence_status: str,\n    required_artifacts: list[Path],\n    missing_artifacts: list[str] | None,\n    safe_wording: str,\n    blocker: str,\n    next_action: str,\n) -> dict:\n    existing = [p.as_posix() for p in required_artifacts if present(p)]\n    return {\n        "claim_id": claim_id,\n        "claim_area": claim_area,\n        "status": status,\n        "manuscript_ready": bool(manuscript_ready),\n        "evidence_status": evidence_status,\n        "required_artifacts": ";".join(p.as_posix() for p in required_artifacts),\n        "existing_artifacts": ";".join(existing),\n        "missing_artifacts": ";".join(missing_artifacts or []),\n        "safe_wording": safe_wording,\n        "blocker": blocker,\n        "next_action": next_action,\n    }\n\n\ndef write_csv(path: Path, rows: list[dict]) -> None:\n    path = resolve(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    fieldnames = [\n        "claim_id",\n        "claim_area",\n        "status",\n        "manuscript_ready",\n        "evidence_status",\n        "required_artifacts",\n        "existing_artifacts",\n        "missing_artifacts",\n        "safe_wording",\n        "blocker",\n        "next_action",\n    ]\n    with path.open("w", newline="", encoding="utf-8") as handle:\n        writer = csv.DictWriter(handle, fieldnames=fieldnames)\n        writer.writeheader()\n        for item in rows:\n            writer.writerow(item)\n\n\ndef main() -> None:\n    args = parse_args()\n    ensure_revision_dirs()\n    print("=" * 80, flush=True)\n    print("CLAIM READINESS GATES", flush=True)\n    print("=" * 80, flush=True)\n\n    transformer_required = [\n        METRIC_DIR / "transformer_ecg_baseline_summary.json",\n        Path("reports/revision/predictions/transformer_ecg_oof_predictions.npz"),\n        MANIFEST_DIR / "transformer_ecg_baseline_manifest.json",\n        METRIC_DIR / "paired_full_vs_transformer_comparison.json",\n        TABLE_DIR / "table_paired_full_vs_transformer.csv",\n        MANIFEST_DIR / "paired_full_vs_transformer_manifest.json",\n    ]\n    transformer_status, transformer_missing = status_from_required(\n        transformer_required,\n        "complete_optional_comparator_available",\n        "not_run_optional_comparator",\n    )\n\n    hybrid_required = [\n        METRIC_DIR / "hybrid_morphology_baseline_summary.json",\n        Path("reports/revision/predictions/hybrid_morphology_oof_predictions.npz"),\n        MANIFEST_DIR / "hybrid_morphology_baseline_manifest.json",\n        METRIC_DIR / "paired_full_vs_hybrid_morphology_comparison.json",\n        TABLE_DIR / "table_paired_full_vs_hybrid_morphology.csv",\n        MANIFEST_DIR / "paired_full_vs_hybrid_morphology_manifest.json",\n    ]\n    hybrid_status, hybrid_missing = status_from_required(\n        hybrid_required,\n        "complete_optional_morphology_sensitivity_available",\n        "not_run_optional_morphology_sensitivity",\n    )\n\n    robustness_required = [\n        METRIC_DIR / "robustness_full_vs_resnet_comparison.json",\n        METRIC_DIR / "robustness_full_vs_raw_mamba_comparison.json",\n        METRIC_DIR / "robustness_multicomparator_pairwise.json",\n        METRIC_DIR / "robustness_multicomparator_summary.csv",\n        TABLE_DIR / "table_robustness_multicomparator.csv",\n        MANIFEST_DIR / "robustness_multicomparator_manifest.json",\n    ]\n    robustness_status, robustness_missing = status_from_required(\n        robustness_required,\n        "complete_multicomparator_robustness_available",\n        "blocked_missing_learned_comparator_stress_evidence",\n    )\n\n    representation_required = [\n        METRIC_DIR / "representation_evidence_status.json",\n        METRIC_DIR / "representation_probe_summary.json",\n        TABLE_DIR / "table_representation_probe.csv",\n        TABLE_DIR / "table_representation_cka.csv",\n        MANIFEST_DIR / "representation_probe_manifest.json",\n    ]\n    representation_status, representation_missing = status_from_required(\n        representation_required,\n        "audit_available_not_mechanistic_proof",\n        "blocked_missing_representation_audit",\n    )\n\n    rows = [\n        row(\n            claim_id="transformer_foundation_baseline",\n            claim_area="Transformer/foundation ECG comparator",\n            status=transformer_status,\n            manuscript_ready=transformer_status.startswith("complete"),\n            evidence_status="optional_comparator_specific",\n            required_artifacts=transformer_required,\n            missing_artifacts=transformer_missing,\n            safe_wording=(\n                "Use only comparator-specific wording if the Transformer ECG baseline and paired bootstrap "\n                "artifacts are complete; otherwise state that this optional comparator was not run."\n            ),\n            blocker="Missing Transformer ECG baseline and/or paired Full-vs-Transformer artifacts."\n            if transformer_missing\n            else "",\n            next_action=(\n                "Run scripts/revision/24_transformer_ecg_baseline.py, then "\n                "scripts/revision/25_paired_full_vs_transformer.py under the frozen OOF contract."\n            ),\n        ),\n        row(\n            claim_id="hybrid_morphology_mlp",\n            claim_area="Hybrid/partially learnable MiniRocket morphology",\n            status=hybrid_status,\n            manuscript_ready=hybrid_status.startswith("complete"),\n            evidence_status="optional_mechanism_sensitivity",\n            required_artifacts=hybrid_required,\n            missing_artifacts=hybrid_missing,\n            safe_wording=(\n                "Use only as morphology-head sensitivity evidence. Do not claim that deterministic "\n                "MiniRocket morphology is causally superior or that the branch is mechanistically isolated."\n            ),\n            blocker="Missing Hybrid MiniRocket-MLP baseline and/or paired comparison artifacts."\n            if hybrid_missing\n            else "",\n            next_action=(\n                "Run scripts/revision/26_hybrid_morphology_baseline.py, then "\n                "scripts/revision/27_paired_full_vs_hybrid_morphology.py."\n            ),\n        ),\n        row(\n            claim_id="robustness_learned_comparators",\n            claim_area="Robustness vs ResNet1D/CNN and Raw Mamba",\n            status=robustness_status,\n            manuscript_ready=robustness_status.startswith("complete"),\n            evidence_status="metric_specific_if_complete",\n            required_artifacts=robustness_required,\n            missing_artifacts=robustness_missing,\n            safe_wording=(\n                "Without learned-comparator stress artifacts, robustness claims remain limited to "\n                "metric-specific Full-vs-MiniRocket evidence."\n            ),\n            blocker="Missing ResNet1D/CNN and/or Raw Mamba stress prediction comparisons."\n            if robustness_missing\n            else "",\n            next_action=(\n                "Save/reuse ResNet and Raw Mamba checkpoints, run comparator stress predictions, then run "\n                "scripts/revision/21_robustness_multicomparator.py."\n            ),\n        ),\n        row(\n            claim_id="full_hrv_feature_set",\n            claim_area="Full HRV feature set",\n            status="blocked_retrain_required",\n            manuscript_ready=False,\n            evidence_status="not_available_for_current_checkpoint",\n            required_artifacts=[],\n            missing_artifacts=["true_RMSSD_SDNN_LFHF_checkpoint_training_contract"],\n            safe_wording=(\n                "Do not claim RMSSD, SDNN, LF/HF, or a complete HRV feature set for the current final-EMA "\n                "checkpoint. The current HRV slots are checkpoint-compatible and partly reserved."\n            ),\n            blocker="Full HRV semantics require a new feature schema and full retraining; they cannot be retrofitted.",\n            next_action="Define a true HRV schema, retrain all folds, regenerate OOF/calibration/baselines, and re-freeze evidence.",\n        ),\n        row(\n            claim_id="mechanistic_disentanglement",\n            claim_area="Mechanistic morphology-rhythm disentanglement",\n            status=representation_status,\n            manuscript_ready=representation_status.startswith("audit_available"),\n            evidence_status="audit_only",\n            required_artifacts=representation_required,\n            missing_artifacts=representation_missing,\n            safe_wording=(\n                "Representation/CKA results may be described only as a conservative branch-embedding audit. "\n                "They do not prove morphology-rhythm disentanglement."\n            ),\n            blocker="" if not representation_missing else "Missing representation probe/CKA artifacts.",\n            next_action=(\n                "If stronger mechanism evidence is required, add preregistered probes with leakage-safe splits and "\n                "keep wording as suggestive unless tests support stronger claims."\n            ),\n        ),\n        row(\n            claim_id="clinical_deployment_readiness",\n            claim_area="Clinical deployment/safety readiness",\n            status="blocked_no_prospective_or_clinical_utility_validation",\n            manuscript_ready=False,\n            evidence_status="not_available",\n            required_artifacts=[],\n            missing_artifacts=["prospective_validation", "clinical_threshold_target", "decision_curve_or_utility_analysis"],\n            safe_wording=(\n                "Do not claim clinical deployment readiness, safety readiness, or prospective clinical utility. "\n                "Current evidence is retrospective/model-evaluation evidence only."\n            ),\n            blocker="No prospective validation, clinical utility analysis, or prespecified deployment threshold target.",\n            next_action="Plan a prospective or clinically curated external validation with utility/threshold analysis.",\n        ),\n        row(\n            claim_id="broad_in_domain_global_superiority",\n            claim_area="Broad in-domain/global superiority",\n            status="contradicted_by_current_fair_baselines",\n            manuscript_ready=False,\n            evidence_status="contradicted",\n            required_artifacts=[\n                METRIC_DIR / "paired_full_vs_resnet_comparison.json",\n                METRIC_DIR / "paired_full_vs_raw_mamba_comparison.json",\n            ],\n            missing_artifacts=[],\n            safe_wording=(\n                "Do not claim SOTA, best in-domain performance, or broad superiority over fair baselines. "\n                "Use metric-specific and comparator-specific wording only."\n            ),\n            blocker="ResNet1D/CNN and Raw Mamba outperform ECG-RAMBA on multiple principal frozen OOF metrics.",\n            next_action="Keep the manuscript framed as structured analysis and evidence-bounded tradeoffs, not global superiority.",\n        ),\n    ]\n\n    out_table = resolve(args.out_table)\n    out_json = resolve(args.out_json)\n    out_manifest = resolve(args.out_manifest)\n    write_csv(out_table, rows)\n    payload = {\n        "status": True,\n        "created_utc": now_utc(),\n        "git_commit": git_commit(),\n        "rows": rows,\n        "claim_guidance": {\n            "use": "Use this table to keep manuscript and rebuttal wording bounded by completed evidence.",\n            "do_not_use": "Do not convert blocked rows into positive claims.",\n        },\n    }\n    save_json(out_json, payload)\n    manifest = {\n        "created_utc": now_utc(),\n        "git_commit": git_commit(),\n        "artifacts": {\n            "json": rel(out_json),\n            "table": rel(out_table),\n        },\n        "artifact_sha256": {\n            "json": sha256_file(out_json),\n            "table": sha256_file(out_table),\n        },\n    }\n    save_json(out_manifest, manifest)\n    print(json.dumps({"status": True, "rows": len(rows), "table": rel(out_table)}, indent=2), flush=True)\n    print(f"Wrote: {out_json}", flush=True)\n    print(f"Wrote: {out_table}", flush=True)\n    print(f"Wrote: {out_manifest}", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'


def _claim_script_is_usable(path: Path) -> bool:
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        text = path.read_text(encoding='utf-8')
    except UnicodeDecodeError:
        return False
    return all(token in text for token in claim_readiness_required_tokens)


def _install_claim_readiness_script_if_needed() -> None:
    if _claim_script_is_usable(claim_readiness_script):
        print(f'Claim readiness script ready: {claim_readiness_script} size={claim_readiness_script.stat().st_size}')
        return

    claim_readiness_script.parent.mkdir(parents=True, exist_ok=True)
    candidate_sources = [
        Path('/content/drive/MyDrive/ECG-Ramba/ECG-RAMBA/scripts/revision/28_claim_readiness_gates.py'),
        Path('/content/drive/MyDrive/ECG-Ramba/revision_artifacts/scripts/revision/28_claim_readiness_gates.py'),
    ]
    for source in candidate_sources:
        if _claim_script_is_usable(source):
            shutil.copy2(source, claim_readiness_script)
            print(f'Restored claim readiness script from: {source}')
            return

    raw_url = 'https://raw.githubusercontent.com/BrianNguyen29/ECG-RAMBA/main/scripts/revision/28_claim_readiness_gates.py'
    try:
        with urllib.request.urlopen(raw_url, timeout=20) as response:
            raw_text = response.read().decode('utf-8')
        if all(token in raw_text for token in claim_readiness_required_tokens):
            claim_readiness_script.write_text(raw_text, encoding='utf-8')
            print(f'Fetched claim readiness script from GitHub raw: {raw_url}')
            return
        print('GitHub raw claim readiness script is stale or incomplete; using embedded fallback.')
    except Exception as exc:
        print(f'Could not fetch claim readiness script from GitHub raw: {exc}; using embedded fallback.')

    claim_readiness_script.write_text(CLAIM_READINESS_FALLBACK_SOURCE, encoding='utf-8')
    if not _claim_script_is_usable(claim_readiness_script):
        raise RuntimeError('Embedded claim readiness script fallback failed validation.')
    print(f'Wrote embedded claim readiness script fallback: {claim_readiness_script} size={claim_readiness_script.stat().st_size}')


_install_claim_readiness_script_if_needed()
run(
    'python -u scripts/revision/28_claim_readiness_gates.py',
    log_path='reports/revision/logs/claim_readiness_gates.log',
)
claim_gate_paths = [
    revision_root / 'metrics' / 'claim_readiness_gates.json',
    revision_root / 'tables' / 'table_claim_readiness_gates.csv',
    revision_root / 'manifests' / 'claim_readiness_gates_manifest.json',
]
for path in claim_gate_paths:
    print(f'{path}: exists={path.exists()} size={path.stat().st_size if path.exists() else None}')


## Final Evidence Matrix

Build claim-level evidence, blocker status, robustness claim rows, and reviewer-safe wording from frozen artifacts.

In [ ]:
import pandas as pd

run(
    'python -u scripts/revision/13_final_evidence_matrix.py --strict',
    log_path='reports/revision/logs/final_evidence_matrix.log',
)

matrix = pd.read_csv('reports/revision/tables/table_final_evidence_matrix.csv')
safe = pd.read_csv('reports/revision/tables/table_final_safe_wording.csv')
robust = pd.read_csv('reports/revision/tables/table_final_robustness_claims.csv')
blockers = pd.read_csv('reports/revision/tables/table_final_blocker_status.csv')
payload = json.loads(Path('reports/revision/metrics/final_evidence_matrix.json').read_text(encoding='utf-8'))

if 'fewshot_artifacts_present' in globals() and fewshot_artifacts_present:
    fewshot_payload = payload.get('fewshot_summary', {})
    if fewshot_payload.get('complete') is not True:
        raise RuntimeError(
            'Few-shot artifacts are present, but final_evidence_matrix.json did not ingest them as complete. '
            'Check scripts/revision/13_final_evidence_matrix.py and rerun this cell.'
        )
    c06_rows = matrix.loc[matrix['claim_id'] == 'C06']
    c06_key_numbers = str(c06_rows.iloc[0]['key_numbers']) if len(c06_rows) else ''
    if 'fewshot_status=complete' not in c06_key_numbers:
        raise RuntimeError('C06 key_numbers does not include fewshot_status=complete; final generator is stale or artifact ingest failed.')
    if 'fewshot' not in payload.get('claim_guidance', {}):
        raise RuntimeError('claim_guidance lacks the fewshot wording block; final generator is stale.')
    print('Few-shot final evidence ingest: OK')
    print('Few-shot complete datasets:', fewshot_payload.get('datasets_complete'))
    print('Few-shot deferred datasets:', fewshot_payload.get('datasets_deferred'))
    print('Few-shot summary:', fewshot_payload.get('key_numbers'))
else:
    print('Few-shot final evidence ingest: deferred because complete few-shot artifacts are absent.')

if (revision_root / 'metrics' / 'claim_readiness_gates.json').exists():
    guidance = payload.get('claim_guidance', {})
    for token in ['claim_readiness_gates', 'hybrid_morphology']:
        if token not in guidance:
            raise RuntimeError(f'Final claim guidance lacks {token}; final generator is stale.')
    print('Claim-readiness final evidence ingest: OK')

print('Evidence matrix:')
display(matrix[['claim_id', 'claim_topic', 'evidence_status', 'key_numbers', 'blocker']])
print('Safe wording:')
display(safe[['claim_id', 'evidence_status', 'safe_wording']])
print('Robustness claim rows:', len(robust))
display(robust[['stress_test', 'metric', 'degradation_interpretation', 'stressed_performance_interpretation']])
print('A0 blocker status:')
display(blockers[['blocker_id', 'resolution_status', 'restriction']])


## Validation Gate

Fail if the final evidence matrix is incomplete or if blocked/limited claims are accidentally marked as fully supported.

In [ ]:
payload = json.loads(Path('reports/revision/metrics/final_evidence_matrix.json').read_text(encoding='utf-8'))
manifest = json.loads(Path('reports/revision/manifests/final_evidence_matrix_manifest.json').read_text(encoding='utf-8'))
required_outputs = [
    Path('reports/revision/metrics/final_evidence_matrix.json'),
    Path('reports/revision/tables/table_final_evidence_matrix.csv'),
    Path('reports/revision/tables/table_final_safe_wording.csv'),
    Path('reports/revision/tables/table_final_blocker_status.csv'),
    Path('reports/revision/tables/table_final_robustness_claims.csv'),
    Path('reports/revision/manifests/final_evidence_matrix_manifest.json'),
]
missing = [str(path) for path in required_outputs if not path.exists()]
if missing:
    raise FileNotFoundError('Missing final evidence outputs: ' + '; '.join(missing))
if payload.get('final_ready_for_rebuttal') is not True:
    raise RuntimeError('Final evidence matrix is not ready for rebuttal use. Inspect unresolved_blockers/contract_issues.')
if payload.get('all_claims_supported') is True:
    raise RuntimeError('Unexpected all_claims_supported=True; blocked/limited claims must remain explicit.')
if len(payload.get('evidence_matrix', [])) != 6:
    raise RuntimeError(f"Final evidence matrix should have 6 claim rows, found {len(payload.get('evidence_matrix', []))}")
if len(payload.get('robustness_claims', [])) != 30:
    raise RuntimeError(f"Final robustness claim table should have 30 rows, found {len(payload.get('robustness_claims', []))}")
if payload.get('missing_inputs'):
    raise RuntimeError('Final evidence matrix reports missing inputs: ' + '; '.join(payload.get('missing_inputs', [])))
if manifest.get('final_ready_for_rebuttal') is not True:
    raise RuntimeError('Final evidence manifest does not mark rebuttal readiness.')
matrix_by_claim = {row.get('claim_id'): row for row in payload.get('evidence_matrix', [])}
c01 = matrix_by_claim.get('C01', {})
c02 = matrix_by_claim.get('C02', {})
c03 = matrix_by_claim.get('C03', {})
c04 = matrix_by_claim.get('C04', {})
c06 = matrix_by_claim.get('C06', {})
if c01.get('source_claim_status') == 'blocked_fair_baselines_missing':
    raise RuntimeError('C01 source_claim_status is stale after Raw Mamba completion.')
for stale_phrase in ['external-transfer, or efficiency evidence', 'If domain AUC is high', 'Robustness rows=']:
    blob = '\n'.join(str(row) for row in payload.get('evidence_matrix', []))
    if stale_phrase in blob:
        raise RuntimeError(f'Stale final wording remains: {stale_phrase}')
if 'ResNet1D/CNN is stronger on PR-AUC, ROC-AUC, F1, Brier, and ECE' not in c02.get('safe_wording', ''):
    raise RuntimeError('C02 safe wording must state the ResNet comparator-specific result.')
if 'near-perfect HRV domain classifier' not in c03.get('safe_wording', ''):
    raise RuntimeError('C03 safe wording must state observed HRV domain sensitivity.')
c04_status = c04.get('evidence_status', '')
c04_numbers = c04.get('key_numbers', '')
c04_safe = c04.get('safe_wording', '')
if c04_status == 'complete_probe_available_with_disentanglement_limitation':
    if 'Probe/CKA complete' not in c04_numbers or 'CKA morphology/rhythm=' not in c04_numbers:
        raise RuntimeError('C04 key_numbers must summarize completed probe/CKA evidence.')
    if 'do not support label-aligned morphology-rhythm disentanglement' not in c04_safe:
        raise RuntimeError('C04 safe wording must forbid disentanglement overclaiming after probe completion.')
elif 'representation separation remains unproven' not in c04_numbers:
    raise RuntimeError('C04 key_numbers must either summarize probe/CKA evidence or keep representation separation unproven.')
external_gate_summary = Path('reports/revision/metrics/external_protocol_gate_summary.csv')
if external_gate_summary.exists():
    gate_df = pd.read_csv(external_gate_summary)
    gate_pass_mask = gate_df.get('protocol_gate_passed', pd.Series(False, index=gate_df.index)).astype(str).str.lower().isin({'true', '1', 'yes'})
    passed_gate_datasets = set(gate_df.loc[gate_pass_mask, 'dataset'].astype(str).str.lower()) if 'dataset' in gate_df.columns else set()
    blocked_gate_datasets = set(gate_df.loc[~gate_pass_mask, 'dataset'].astype(str).str.lower()) if 'dataset' in gate_df.columns else set()
    expected_external = {'ptbxl', 'georgia', 'cpsc2021'}
    deferred_gate_datasets = expected_external - passed_gate_datasets - blocked_gate_datasets
    expected_passed_text = ','.join(sorted(passed_gate_datasets)) if passed_gate_datasets else 'none'
    expected_blocked_text = ','.join(sorted(blocked_gate_datasets)) if blocked_gate_datasets else 'none'
    expected_deferred_text = ','.join(dataset for dataset in ['ptbxl', 'georgia', 'cpsc2021'] if dataset in deferred_gate_datasets) or 'none'
    c06_key_numbers = c06.get('key_numbers', '')
    if gate_pass_mask.any() and 'protocol_gated' not in c06.get('evidence_status', ''):
        raise RuntimeError('C06 evidence_status must reflect protocol-gated external datasets after a gate pass.')
    if f'external_gate_passed={expected_passed_text}' not in c06_key_numbers:
        raise RuntimeError(f'C06 key_numbers must list passed external gates: {expected_passed_text}')
    if f'external_gate_blocked={expected_blocked_text}' not in c06_key_numbers:
        raise RuntimeError(f'C06 key_numbers must list blocked external gates: {expected_blocked_text}')
    if f'external_gate_deferred={expected_deferred_text}' not in c06_key_numbers:
        raise RuntimeError(f'C06 key_numbers must list deferred external gates: {expected_deferred_text}')
    c06_safe_wording = c06.get('safe_wording', '')
    if 'Protocol-gated mapped-task external evaluation is available only for:' not in c06_safe_wording:
        raise RuntimeError('C06 safe wording must list protocol-gated passed external datasets.')
    if 'No unqualified external-transfer or cross-dataset performance-advantage claim is supported' not in c06_safe_wording:
        raise RuntimeError('C06 safe wording must explicitly restrict unqualified external-transfer/cross-dataset performance-advantage claims.')
print('Final evidence matrix validated.')
print('All claims supported:', payload.get('all_claims_supported'))
print('Contract issues:', payload.get('contract_issues'))


## Inventory And Stable Mirror

Refresh the artifact manifest and publish final evidence outputs back to Drive.

In [ ]:
run(
    'python -u scripts/revision/05_artifact_inventory.py',
    log_path='reports/revision/logs/final_artifact_inventory.log',
)
run(
    f'python -u scripts/revision/artifact_mirror.py publish --mirror-root "{MIRROR_REVISION_ROOT}"',
    log_path='reports/revision/logs/final_evidence_mirror_publish.log',
)


## Convenience Drive Copies

Copy final reviewer tables to a short Drive path for manual download/opening. These are convenience copies; the canonical artifacts remain under `reports/revision/` and the verified mirror.

In [ ]:
import shutil

FINAL_TABLE_EXPORT_DIR = DRIVE_ROOT / 'final_evidence_tables'
FINAL_TABLE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
final_table_sources = [
    Path('reports/revision/metrics/final_evidence_matrix.json'),
    Path('reports/revision/tables/table_final_evidence_matrix.csv'),
    Path('reports/revision/tables/table_final_safe_wording.csv'),
    Path('reports/revision/tables/table_final_blocker_status.csv'),
    Path('reports/revision/tables/table_final_robustness_claims.csv'),
    Path('reports/revision/manifests/final_evidence_matrix_manifest.json'),
]
optional_table_sources = [
    Path('reports/revision/metrics/external_protocol_gate_summary.csv'),
    Path('reports/revision/metrics/external_ptbxl_protocol_gate.json'),
    Path('reports/revision/tables/table_external_ptbxl_label_mapping.csv'),
    Path('reports/revision/tables/table_external_ptbxl_metrics.csv'),
    Path('reports/revision/manifests/external_ptbxl_protocol_gate_manifest.json'),
    Path('reports/revision/metrics/external_georgia_protocol_gate.json'),
    Path('reports/revision/tables/table_external_georgia_label_mapping.csv'),
    Path('reports/revision/tables/table_external_georgia_metrics.csv'),
    Path('reports/revision/manifests/external_georgia_protocol_gate_manifest.json'),
    Path('reports/revision/metrics/external_cpsc2021_protocol_gate.json'),
    Path('reports/revision/tables/table_external_cpsc2021_label_mapping.csv'),
    Path('reports/revision/tables/table_external_cpsc2021_metrics.csv'),
    Path('reports/revision/manifests/external_cpsc2021_protocol_gate_manifest.json'),
    Path('reports/revision/metrics/representation_evidence_status.json'),
    Path('reports/revision/metrics/representation_probe_summary.json'),
    Path('reports/revision/tables/table_representation_probe.csv'),
    Path('reports/revision/tables/table_representation_cka.csv'),
    Path('reports/revision/manifests/representation_probe_manifest.json'),
    Path('reports/revision/metrics/fewshot_ptbxl_summary.csv'),
    Path('reports/revision/tables/table_fewshot_ptbxl.csv'),
    Path('reports/revision/metrics/fewshot_ptbxl_bootstrap.json'),
    Path('reports/revision/manifests/fewshot_ptbxl_run_manifest.json'),
    Path('reports/revision/metrics/fewshot_cpsc2021_summary.csv'),
    Path('reports/revision/tables/table_fewshot_cpsc2021.csv'),
    Path('reports/revision/metrics/fewshot_cpsc2021_bootstrap.json'),
    Path('reports/revision/manifests/fewshot_cpsc2021_run_manifest.json'),
    Path('reports/revision/metrics/fewshot_georgia_summary.csv'),
    Path('reports/revision/tables/table_fewshot_georgia.csv'),
    Path('reports/revision/metrics/fewshot_georgia_bootstrap.json'),
    Path('reports/revision/manifests/fewshot_georgia_run_manifest.json'),
    Path('reports/revision/metrics/robustness_multicomparator_summary.csv'),
    Path('reports/revision/metrics/robustness_multicomparator_pairwise.json'),
    Path('reports/revision/tables/table_robustness_multicomparator.csv'),
    Path('reports/revision/manifests/robustness_multicomparator_manifest.json'),
    Path('reports/revision/metrics/claim_readiness_gates.json'),
    Path('reports/revision/metrics/hybrid_morphology_baseline_summary.json'),
    Path('reports/revision/manifests/paired_full_vs_hybrid_morphology_manifest.json'),
    Path('reports/revision/tables/table_paired_full_vs_hybrid_morphology.csv'),
    Path('reports/revision/metrics/paired_full_vs_hybrid_morphology_comparison.json'),
    Path('reports/revision/manifests/claim_readiness_gates_manifest.json'),
    Path('reports/revision/tables/table_claim_readiness_gates.csv'),
]
for source in final_table_sources:
    if not source.exists():
        raise FileNotFoundError(source)
    destination = FINAL_TABLE_EXPORT_DIR / source.name
    shutil.copy2(source, destination)
    print(f'Copied: {destination} size={destination.stat().st_size}')

print('Optional convenience copies:')
for source in optional_table_sources:
    if source.exists():
        destination = FINAL_TABLE_EXPORT_DIR / source.name
        shutil.copy2(source, destination)
        print(f'Copied optional: {destination} size={destination.stat().st_size}')
    else:
        print(f'Skipped optional missing: {source}')

readme = FINAL_TABLE_EXPORT_DIR / 'README.md'
readme.write_text(
    '# ECG-RAMBA final evidence tables\n\n'
    'Convenience copies generated by notebooks/07_results_freeze.ipynb.\n\n'
    '- table_final_evidence_matrix.csv: claim-level evidence status and key numbers.\n'
    '- table_final_safe_wording.csv: reviewer/manuscript-safe wording.\n'
    '- table_final_blocker_status.csv: A0 blocker decisions and restrictions.\n'
    '- table_final_robustness_claims.csv: stress/metric-specific robustness CI interpretations.\n'
    '- final_evidence_matrix.json: full machine-readable payload.\n'
    '- external_protocol_gate_summary.csv and table_external_* files: protocol-gated mapped-task external evidence copied when present. For the current evidence package, PTB-XL, Georgia, and CPSC2021 pass their dataset-specific gates; all remain mapped-task evidence only.\n'
    '- representation_probe_summary.json, table_representation_probe.csv, and table_representation_cka.csv: optional representation audit copied when present; use only as suggestive/limitation evidence, not proof of disentanglement.\n'
    '- fewshot_<dataset>_* files: gated score-calibration sensitivity analyses, not model-weight updating. PTB-XL, Georgia, and CPSC2021 are complete when their files are present.\n'
    '- robustness_multicomparator_* files: optional ledger for robustness beyond MiniRocket; missing ResNet/Raw-Mamba stress predictions remain blocked rows.\n'
    '- claim_readiness_gates.json and table_claim_readiness_gates.csv: blocker ledger for optional or unsupported claims; do not convert blocked rows into positive claims.\n\n'
    'Canonical verified artifacts remain under reports/revision and the revision_artifacts mirror.\n',
    encoding='utf-8',
)
print('Convenience export dir:', FINAL_TABLE_EXPORT_DIR)


## Output Summary

In [ ]:
expected = [
    'reports/revision/metrics/final_evidence_matrix.json',
    'reports/revision/tables/table_final_evidence_matrix.csv',
    'reports/revision/tables/table_final_safe_wording.csv',
    'reports/revision/tables/table_final_blocker_status.csv',
    'reports/revision/tables/table_final_robustness_claims.csv',
    'reports/revision/manifests/final_evidence_matrix_manifest.json',
    'reports/revision/manifests/artifacts_manifest.json',
    'reports/revision/manifests/artifacts_manifest.csv',
    str(DRIVE_ROOT / 'final_evidence_tables' / 'table_final_evidence_matrix.csv'),
    str(DRIVE_ROOT / 'final_evidence_tables' / 'table_final_safe_wording.csv'),
]
for item in expected:
    path = Path(item)
    print(f'{item}: exists={path.exists()} size={path.stat().st_size if path.exists() else None}')

optional_expected = [
    'reports/revision/metrics/paired_full_vs_raw_mamba_comparison.json',
    'reports/revision/tables/table_paired_full_vs_raw_mamba.csv',
    'reports/revision/manifests/paired_full_vs_raw_mamba_manifest.json',
    'reports/revision/metrics/paired_full_vs_transformer_comparison.json',
    'reports/revision/tables/table_paired_full_vs_transformer.csv',
    'reports/revision/manifests/paired_full_vs_transformer_manifest.json',
    'reports/revision/metrics/external_protocol_gate_summary.csv',
    'reports/revision/metrics/external_ptbxl_protocol_gate.json',
    'reports/revision/tables/table_external_ptbxl_label_mapping.csv',
    'reports/revision/tables/table_external_ptbxl_metrics.csv',
    'reports/revision/manifests/external_ptbxl_protocol_gate_manifest.json',
    'reports/revision/metrics/external_georgia_protocol_gate.json',
    'reports/revision/tables/table_external_georgia_label_mapping.csv',
    'reports/revision/tables/table_external_georgia_metrics.csv',
    'reports/revision/manifests/external_georgia_protocol_gate_manifest.json',
    'reports/revision/metrics/external_cpsc2021_protocol_gate.json',
    'reports/revision/tables/table_external_cpsc2021_label_mapping.csv',
    'reports/revision/tables/table_external_cpsc2021_metrics.csv',
    'reports/revision/manifests/external_cpsc2021_protocol_gate_manifest.json',
    'reports/revision/metrics/representation_evidence_status.json',
    'reports/revision/metrics/representation_probe_summary.json',
    'reports/revision/tables/table_representation_probe.csv',
    'reports/revision/tables/table_representation_cka.csv',
    'reports/revision/manifests/representation_probe_manifest.json',
    'reports/revision/metrics/fewshot_ptbxl_summary.csv',
    'reports/revision/tables/table_fewshot_ptbxl.csv',
    'reports/revision/metrics/fewshot_ptbxl_bootstrap.json',
    'reports/revision/manifests/fewshot_ptbxl_run_manifest.json',
    'reports/revision/metrics/robustness_multicomparator_summary.csv',
    'reports/revision/metrics/robustness_multicomparator_pairwise.json',
    'reports/revision/tables/table_robustness_multicomparator.csv',
    'reports/revision/manifests/robustness_multicomparator_manifest.json',
    str(DRIVE_ROOT / 'final_evidence_tables' / 'external_protocol_gate_summary.csv'),
    str(DRIVE_ROOT / 'final_evidence_tables' / 'table_external_ptbxl_label_mapping.csv'),
    str(DRIVE_ROOT / 'final_evidence_tables' / 'table_external_ptbxl_metrics.csv'),
    'reports/revision/metrics/claim_readiness_gates.json',
    str(DRIVE_ROOT / 'final_evidence_tables' / 'table_claim_readiness_gates.csv'),
    str(DRIVE_ROOT / 'final_evidence_tables' / 'claim_readiness_gates.json'),
    'reports/revision/metrics/hybrid_morphology_baseline_summary.json',
    'reports/revision/manifests/paired_full_vs_hybrid_morphology_manifest.json',
    'reports/revision/tables/table_paired_full_vs_hybrid_morphology.csv',
    'reports/revision/metrics/paired_full_vs_hybrid_morphology_comparison.json',
    'reports/revision/manifests/claim_readiness_gates_manifest.json',
    'reports/revision/tables/table_claim_readiness_gates.csv',
]
print('\nOptional evidence artifacts:')
for item in optional_expected:
    path = Path(item)
    print(f'{item}: exists={path.exists()} size={path.stat().st_size if path.exists() else None}')

payload = json.loads(Path('reports/revision/metrics/final_evidence_matrix.json').read_text(encoding='utf-8'))
print('\nFinal guidance:')
for key, value in payload.get('claim_guidance', {}).items():
    print(f'- {key}: {value}')
print('\nFew-shot final summary:')
print(payload.get('fewshot_summary', {}))
print('\nConvenience Drive folder:', DRIVE_ROOT / 'final_evidence_tables')
print('Notebook 07 complete. Use table_final_evidence_matrix.csv and table_final_safe_wording.csv as the manuscript/rebuttal source of truth.')
